In [15]:
import token


class ByteTokenizer():
    """Represent a string as a sequence of bytes."""
    def encode(self, string: str) -> list[int]:
        string_bytes = string.encode("utf-8")
        indices = list(map(int, string_bytes))
        return indices
    def decode(self, indices: list[int]) -> str:
        string_bytes = bytes(indices)  
        string = string_bytes.decode("utf-8")  
        return string
    
token = ByteTokenizer()

print(token.encode("大"))
print(token.encode("家"))
print(token.encode("吃"))

print(token.encode("大家好"))
# token.encode("大家好")

[229, 164, 167]
[229, 174, 182]
[229, 144, 131]
[229, 164, 167, 229, 174, 182, 229, 165, 189]


In [16]:
def merge(indices: list[int], pair: tuple[int, int], new_index: int) -> list[int]:  # @inspect indices, @inspect pair, @inspect new_index
    """Return `indices`, but with all instances of `pair` replaced with `new_index`."""
    new_indices = []  # @inspect new_indices
    i = 0  # @inspect i
    while i < len(indices):
        if i + 1 < len(indices) and indices[i] == pair[0] and indices[i + 1] == pair[1]:
            new_indices.append(new_index)
            i += 2
        else:
            new_indices.append(indices[i])
            i += 1
    return new_indices

In [17]:
import os

from collections import Counter
from pathlib import Path
from typing import Dict, List, Tuple

# ---------------------------------------------------------------------------
# Type aliases
Token = int                # internal numeric id
Subword = bytes            # raw byte sequence
Pair   = Tuple[int, int]   # adjacent token pair


class BPETokenizer:
    """Pure‑Python BPE tokenizer ― now with special‑token capability."""

    # ------------------------------------------------------------------
    # Initialization
    # ------------------------------------------------------------------
    def __init__(
        self,
        vocab: Dict[int, Subword] | None = None,
        merges: List[Pair] | None = None,
        special_tokens: List[str] | None = None,
    ) -> None:
        # 基础词表与 merge 规则
        self.vocab: Dict[int, Subword] = vocab or {}
        self.merges: List[Pair]         = merges or []

        # special token 双向映射（字符串 <-> id）
        self.special_tokens: List[str]          = special_tokens or []
        self.special_token_to_id: Dict[str, int] = {}
        self.id_to_special_token: Dict[int, str] = {}

        # 反向字节查词表
        self._inv_vocab: Dict[Subword, int] = {b: i for i, b in self.vocab.items()}

        # 确保 special token 已插入 vocab
        for tok in self.special_tokens:
            self.add_special_token(tok)

    # ------------------------------------------------------------------
    # Public encode / decode API (签名保持不变)
    # ------------------------------------------------------------------
    def encode(self, string: str) -> List[int]:
        """将 *string* 转成 token id 序列。

        * 若输入整段即为注册的 special token（如 "<pad>"），直接返回对应 id。
        * 否则按字节拆分并应用已学习的 merge 规则。
        """
        # 特殊情况：整段是一个 special token
        if string in self.special_token_to_id:
            return [self.special_token_to_id[string]]

        # 1) 字节级拆分
        indices = [self._inv_vocab[bytes([b])] for b in string.encode("utf-8")]
        # 2) 依次执行 merge 规则
        for pair in self.merges:
            indices = self.merge(indices, pair, self.pair_to_id(pair))
        return indices

    def decode(self, indices: List[int]) -> str:
        """将 token id 序列还原为人类可读文本。
        special id 会被直接替换成其字符串（如 "<eos>")。
        """
        if not indices:
            return ""
        parts: List[str] = []
        for idx in indices:
            if idx in self.id_to_special_token:
                parts.append(self.id_to_special_token[idx])
            else:
                parts.append(self.vocab[idx].decode("utf-8", errors="replace"))
        return "".join(parts)

    
    @staticmethod
    def merge(indices: List[int], pair: Pair, new_index: int) -> List[int]:
        new_indices: List[int] = []
        i = 0
        while i < len(indices):
            if i + 1 < len(indices) and indices[i] == pair[0] and indices[i + 1] == pair[1]:
                new_indices.append(new_index)
                i += 2
            else:
                new_indices.append(indices[i])
                i += 1
        return new_indices

    @staticmethod
    def calc_freq(dataset: List[List[int]]) -> Counter[Pair]:
        freq: Counter[Pair] = Counter()
        for seq in dataset:
            for a, b in zip(seq, seq[1:]):
                freq[(a, b)] += 1
        return freq

    def train_from_file(self, file_path: str | Path, target_vocab_size: int = 30000) -> None:
        """Load corpus and train *in‑place* until vocab size reaches *target_vocab_size*."""
        text = Path(file_path).read_text(encoding="utf-8")

        # 1) ensure byte‑level base vocab exists
        if not self.vocab:
            for b in sorted(set(text.encode("utf-8"))):
                self.add_token(bytes([b]))

        dataset = [self.encode(story) for story in text.split('<|endoftext|>')]

        # 2) iterative BPE merges
        while len(self.vocab) < target_vocab_size:
            pair_freq = self.calc_freq(dataset)
            if not pair_freq:
                break
            best_pair, _ = max(pair_freq.items(), key=lambda kv: kv[1])
            new_id = self.add_token(self.vocab[best_pair[0]] + self.vocab[best_pair[1]])
            self.merges.append(best_pair)
            dataset = [self.merge(seq, best_pair, new_id) for seq in dataset]
    
    
    def add_token(self, subword: Subword) -> Token:
        if subword in self._inv_vocab:
            return self._inv_vocab[subword]
        idx = len(self.vocab)
        self.vocab[idx] = subword
        self._inv_vocab[subword] = idx
        return idx

   

    def pair_to_id(self, pair: Pair) -> int:
        return self._inv_vocab[self.vocab[pair[0]] + self.vocab[pair[1]]]